<h1 style="text-align: center;">Cafe Sales Analysis: Dirty Dataset Cleaning</h1>
<!-- <h3 style="font-style: italic; text-align: center;">Dirty Dataset Cleaning</h3> -->

In [19]:
from pathlib import Path

import numpy as np
import pandas as pd

---

## Data Description

> The Dirty Cafe Sales dataset contains 10,000 rows and 8 columns of synthetic data representing sales transactions in a cafe. This dataset is intentionally "dirty", with missing values, inconsistent data, and errors introduced to provide a realistic scenario for data cleaning and exploratory data analysis (EDA). It can be used—especially as the main focus of this notebook—to practice cleaning techniques and data wrangling.

*Adapted from [Cafe Sales - Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training) by [Ahmed Mohamed](https://www.kaggle.com/ahmedmohamed2003).*

### File Information

* **File Name:** `dirty_cafe_sales.csv`
* **Number of Rows:** 10,000
* **Number of Columns:** 8

### Columns Description

| Column Name | Description | Example Values |
| --- | --- | --- |
| `Transaction ID` | A unique identifier for each transaction. Always present and unique. | `TXN_1234567` |
| `Item` | The name of the item purchased. May contain missing or invalid values (e.g., "ERROR"). | `Coffee, Sandwich` |
| `Quantity` | The quantity of the item purchased. May contain missing or invalid values. | `1, 3, UNKNOWN` |
| `Price Per Unit` | The price of a single unit of the item. May contain missing or invalid values. | `2.00, 4.00` |
| `Total Spent` | The total amount spent on the transaction. Calculated as `Quantity * Price Per Unit`. | `8.00, 12.00` |
| `Payment Method` | The method of payment used. May contain missing or invalid values (e.g., None, "UNKNOWN"). | `Cash, Credit Card` |
| `Location` | The location where the transaction occurred. May contain missing or invalid values. | `In-store, Takeaway` |
| `Transaction Date` | The date of the transaction. May contain missing or incorrect values. | `2023-01-01` |

### Data Characteristics

1. **Missing Values**: Some columns (e.g., Item, Payment Method, Location) may contain missing values represented as None or empty cells.
2. **Invalid Values**: Some rows contain invalid entries like "ERROR" or "UNKNOWN" to simulate real-world data issues.
3. **Price Consistency**: Prices for menu items are consistent but may have missing or incorrect values introduced.
* The dataset includes the following menu items with their respective price ranges:

    | Item | Price ($) |
    | --- | :---: |
    | Coffee | 2 |
    | Tea | 1.5 |
    | Sandwich | 4 |
    | Salad | 5 |
    | Cake | 3 |
    | Cookie | 1 |
    | Smoothie | 4 |
    | Juice | 3 |  

---

## Metadata Analysis

In [20]:
# get path to the notebook and raw data file
notebook_path = Path.cwd()
raw_data_path = notebook_path.parent / "data" / "raw" / "dirty_cafe_sales.csv"

# read data file and display a sample of 5 rows
raw_data = pd.read_csv(raw_data_path)
raw_data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [21]:
# reproduce pd.info() with additional information about unique values in each column
with pd.option_context("display.max_colwidth", 100):
    display(
        pd.DataFrame(
            {
                "Columns": raw_data.columns,
                "Non-Null Count": raw_data.notna().sum().values,
                "Dtype": raw_data.dtypes.values,
                "Unique Values": [raw_data[column].unique() for column in raw_data.columns]
            }
        )
    )

,Columns,Non-Null Count,Dtype,Unique Values
0,Transaction ID,10000,str,"[TXN_1961373, TXN_4977031, TXN_4271903, TXN_7034554, TXN_3160411, TXN_2602893, TXN_4433211, TXN_..."
1,Item,9667,str,"[Coffee, Cake, Cookie, Salad, Smoothie, UNKNOWN, Sandwich, nan, ERROR, Juice, Tea]"
2,Quantity,9862,str,"[2, 4, 5, 3, 1, ERROR, UNKNOWN, nan]"
3,Price Per Unit,9821,str,"[2.0, 3.0, 1.0, 5.0, 4.0, 1.5, nan, ERROR, UNKNOWN]"
4,Total Spent,9827,str,"[4.0, 12.0, ERROR, 10.0, 20.0, 9.0, 16.0, 15.0, 25.0, 8.0, 5.0, 3.0, 6.0, nan, UNKNOWN, 2.0, 1.0..."
5,Payment Method,7421,str,"[Credit Card, Cash, UNKNOWN, Digital Wallet, ERROR, nan]"
6,Location,6735,str,"[Takeaway, In-store, UNKNOWN, nan, ERROR]"
7,Transaction Date,9841,str,"[2023-09-08, 2023-05-16, 2023-07-19, 2023-04-27, 2023-06-11, 2023-03-31, 2023-10-06, 2023-10-28,..."


<table>
  <tr>
    <th style="width:10rem">Category</th>
    <th>Insight</th>
  </tr>
  <tr>
    <td>Data Dimensionality</td>
    <td>Jumlah <code>Columns</code> serta <code>Non-Null Count</code> dari <code>Transaction ID</code> sudah sesuai dengan penjabaran pada <a href="#data-description">Data Description</a>: <code>8</code> variables over <code>10000</code> unique observations.</td>
  </tr>
  <tr>
    <td><b>Missing Data</b></td>
    <td><b>Missing values teridentifikasi dari <code>Non-Null Count</code> pada seluruh <code>Columns</code> selain <code>Transaction ID</code>, yang direpresentasikan dengan nilai <code>nan</code> pada <code>Unique Values</code> dan jumlahnya bervariasi.</b></td>
  </tr>
  <tr>
    <td rowspan="3"><b>Invalid Data Type</b></td>
    <td><b><code>Quantity</code> sebagai jumlah purchased menu items seharusnya bertipe <code>int</code>.</b></td>
  </tr>
  <tr>
    <td><b><code>Price Per Unit</code> of menu items yang salah satunya memiliki decimal point (<code>1.5</code>) seharusnya bertipe <code>float</code>. Pun demikian untuk <code>Total Spent</code> sebagai hasil perkalian antara <code>Quantity (int)</code> dengan <code>Price Per Unit (float)</code>.</b></td>
  </tr>
  <tr>
    <td><b><code>Transaction Date</code> akan diubah to the appropriate <code>datetime</code> type.</b></td>
  </tr>
  <tr>
    <td><b>Invalid Values</b></td>
    <td><b>Invalid entries detected on several <code>Columns</code>, such as <code>ERROR</code> and <code>UNKNOWN</code>.</b></td>
  </tr>
</table>

---

## Data Cleaning

>To clean this dataset, consider the following steps:
>
>1. Handle Missing Values:
>    * Fill missing numeric values with the median or mean.
>    * Replace missing categorical values with the mode or "Unknown."
>
>2. Handle Invalid Values:
>    * Replace invalid entries like "ERROR" and "UNKNOWN" with NaN or appropriate values.
>
>3. Date Consistency:
>    * Ensure all dates are in a consistent format.
>    * Fill missing dates with plausible values based on nearby records.
>
>4. Feature Engineering:
>    * Create new columns, such as Day of the Week or Transaction Month, for further analysis.

### Invalid Values

* Following on suggested data cleaning processes, invalid entries may be replaced with `nan` or the appropriate values.
    * Sehingga tahapan ini didahulukan sebelum melakukan imputasi terhadap missing values.
* Starting from invalid values on the first column `Item`, previous section on [Data Characteristics](#data-characteristics) provides information regarding a list of menu items which uniquely corresponds to their `Price Per Unit`.
    * For example, `coffee` is the only one on the list with the `Price ($)` of `2.0`.
        * Sehingga replacing invalid values on these columns bisa dilakukan dengan maintaining a kind of "consistency" between them. 
        * E.g. `ERROR` entries in `Item` with `Price Per Unit` of `2.0` is certain to be `Coffee`, and vice versa.
    * With the exceptions of `Sandwich` which shares the same `Price Per Unit` of `4.0` with `Smoothie`, and `Cake` with `Juice (3.0)`.
        * The replacement strategy may only be implemented in "one-way".
        * I.e. `Sandwich & Smoothie` have `Price Per Unit = 4.0`, but not all observations with `Price Per Unit = 4.0` is either `Sandwich / Smoothie`.
* Berikutnya, invalid entries yang berwujud `UNKNOWN` ataupun `ERROR` diganti dengan missing value `nan` agar kemudian dapat dilakukan imputasi.

In [ ]:
# define conditions and choices for replacing values
condlist_item = [
    (raw_data["Price Per Unit"] == "2.0"),
    (raw_data["Price Per Unit"] == "1.5"),
    (raw_data["Price Per Unit"] == "5.0"),
    (raw_data["Price Per Unit"] == "1.0")
]
choicelist_item = [
    "Coffee",
    "Tea",
    "Salad",
    "Cookie"
]
condlist_price_per_unit = [
    (raw_data["Item"] == "Coffee"),
    (raw_data["Item"] == "Tea"),
    (raw_data["Item"] == "Sandwich"),
    (raw_data["Item"] == "Salad"),
    (raw_data["Item"] == "Cake"),
    (raw_data["Item"] == "Cookie"),
    (raw_data["Item"] == "Smoothie"),
    (raw_data["Item"] == "Juice")
]
choicelist_price_per_unit = [
    "2.0",
    "1.5",
    "4.0",
    "5.0",
    "3.0",
    "1.0",
    "4.0",
    "3.0"
]
condlist_invalid_values = [
    ((raw_data[column] == "UNKNOWN") | (raw_data[column] == "ERROR"))
    for column in raw_data.columns
]
choicelist_invalid_values = [
    np.nan for _ in raw_data.columns
]

# replace values according to the defined conditions and choices
cleaned_data = raw_data.copy()
cleaned_data["Item"] = np.select(
    condlist_item,
    choicelist_item,
    default=cleaned_data["Item"]
)
cleaned_data["Price Per Unit"] = np.select(
    condlist_price_per_unit,
    choicelist_price_per_unit,
    default=cleaned_data["Price Per Unit"]
)
for index, column in enumerate(raw_data.columns):
    cleaned_data[column] = np.select(
        [condlist_invalid_values[index]],
        [choicelist_invalid_values[index]],
        default=cleaned_data[column]
    )

### Missing Values

In [25]:
# reproduce pd.info() with additional information about unique values in each column
with pd.option_context("display.max_colwidth", 100):
    display(
        pd.DataFrame(
            {
                "Columns": cleaned_data.columns,
                "Non-Null Count": cleaned_data.notna().sum().values,
                "Dtype": cleaned_data.dtypes.values,
                "Unique Values": [cleaned_data[column].unique() for column in cleaned_data.columns]
            }
        )
    )

,Columns,Non-Null Count,Dtype,Unique Values
0,Transaction ID,10000,str,"[TXN_1961373, TXN_4977031, TXN_4271903, TXN_7034554, TXN_3160411, TXN_2602893, TXN_4433211, TXN_..."
1,Item,9179,str,"[Coffee, Cake, Cookie, Salad, Smoothie, nan, Sandwich, Juice, Tea]"
2,Quantity,9521,str,"[2, 4, 5, 3, 1, nan]"
3,Price Per Unit,9625,str,"[2.0, 3.0, 1.0, 5.0, 4.0, 1.5, nan]"
4,Total Spent,9498,str,"[4.0, 12.0, nan, 10.0, 20.0, 9.0, 16.0, 15.0, 25.0, 8.0, 5.0, 3.0, 6.0, 2.0, 1.0, 7.5, 4.5, 1.5]"
5,Payment Method,6822,str,"[Credit Card, Cash, nan, Digital Wallet]"
6,Location,6039,str,"[Takeaway, In-store, nan]"
7,Transaction Date,9540,str,"[2023-09-08, 2023-05-16, 2023-07-19, 2023-04-27, 2023-06-11, 2023-03-31, 2023-10-06, 2023-10-28,..."


In [ ]:
# display(
#     raw_data["Item"]\
#         .value_counts()\
#             .to_frame(name="Count")\
#                 .sort_values(by="Count", ascending=False)
# )
# condlist_item = [
#     (raw_data["Item"].isna())
# ]
# choicelist_item = [
#     raw_data["Item"].mode()[0]
# ]
# filter and display count of items with certain price per unit
# print("Count of Item with Price Per Unit = 4.0")
# display(
#     raw_data[raw_data["Price Per Unit"] == "4.0"]["Item"]\
#         .value_counts()\
#             .to_frame(name="Count")\
#                 .sort_values(by="Count", ascending=False)
# )
# print("Count of Item with Price Per Unit = 3.0")
# display(
#     raw_data[raw_data["Price Per Unit"] == "3.0"]["Item"]\
#         .value_counts()\
#             .to_frame(name="Count")\
#                 .sort_values(by="Count", ascending=False)
# )
# * Analisis berikutnya menampilkan counts of `Item` with the same value of `Price Per Unit`.